# Travel Reimbursement Approval Agent



This notebook implements a complete Travel Reimbursement Approval Agent. It reviews the five mock claims supplied in the assignment against the provided travel policy, applies receipt and spending-limit checks, and returns a structured recommendation for every claim.

I separate policy calculations from language generation. The policy engine determines amounts and decision status in a repeatable way, while the LLM is used for grounded, tool-based explanation.

## README — Setup and how to run

1. Run the installation cell once in a fresh Python environment.
2. Run the remaining cells from top to bottom.
3. The policy workflow, validation checks, results table, and dashboard run without an API key.
4. To demonstrate LLM tool calling, set GROQ_API_KEY in the environment before starting Jupyter or VS Code. 


In [1]:
%pip install -q langchain-core langchain-groq pydantic pandas plotly


[notice] A new release of pip available: 22.2.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Policy context and claim intake

In [2]:
import json,os,re
from datetime import date
from typing import Literal
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, display
from pydantic import BaseModel,Field
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage,SystemMessage,ToolMessage

GROQ_API_KEY = os.getenv('GROQ_API_KEY')

POLICIES={'POL-CAT-01':'Eligible: economy airfare, lodging, meals, ground transport, conference fees.','POL-CAT-02':'Alcohol, minibar, spa, gym, entertainment, shopping, gifts, fines, penalties, late fees and personal expenses are never reimbursable.','POL-PD-01':'Meals: $75/day.','POL-PD-02':'Lodging: $200/night.','POL-PD-03':'Ground transport: $50/day.','POL-AIR-01':'Only economy airfare is reimbursable; business/first class requires Manual Review.','POL-RCT-01':'Items over $25, plus all airfare and lodging, require itemized receipts.','POL-RCT-02':'Missing required receipts require Manual Review.','POL-APR-01':'Totals <= $500 may auto-approve.','POL-APR-02':'Totals > $500 and <= $2,000 are manager-approvable.','POL-APR-03':'Totals > $2,000 require Manual Review.','POL-TIME-01':'Claims later than 30 days require Manual Review.'}
CLAIMS=[
{'claim_id':'CLM-001','employee':'A. Rivera','trip_start':'2026-06-10','trip_end':'2026-06-12','submitted':'2026-06-20','items':[{'category':'airfare','description':'Round-trip economy airfare','amount':420.,'receipt':True},{'category':'lodging','description':'Hotel, 2 nights @ $180','amount':360.,'receipt':True},{'category':'meals','description':'Meals, 3 days @ ~$60/day','amount':180.,'receipt':True},{'category':'conference_fees','description':'Conference registration','amount':150.,'receipt':True}]},
{'claim_id':'CLM-002','employee':'B. Osei','trip_start':'2026-06-14','trip_end':'2026-06-15','submitted':'2026-06-25','items':[{'category':'spa','description':'Hotel spa package','amount':300.,'receipt':True},{'category':'minibar','description':'In-room minibar','amount':80.,'receipt':True}]},
{'claim_id':'CLM-003','employee':'C. Nakamura','trip_start':'2026-06-08','trip_end':'2026-06-10','submitted':'2026-06-22','items':[{'category':'airfare','description':'Round-trip economy airfare','amount':300.,'receipt':True},{'category':'lodging','description':'Hotel, 2 nights @ $250','amount':500.,'receipt':True},{'category':'meals','description':'Meals, 2 days @ $70/day','amount':140.,'receipt':True}]},
{'claim_id':'CLM-004','employee':'D. Fischer','trip_start':'2026-06-16','trip_end':'2026-06-18','submitted':'2026-06-28','items':[{'category':'airfare','description':'Business-class international airfare','amount':2400.,'receipt':True},{'category':'lodging','description':'Hotel, 3 nights','amount':600.,'receipt':False}]},
{'claim_id':'CLM-005','employee':'E. Haddad','trip_start':'2026-06-11','trip_end':'2026-06-11','submitted':'2026-06-24','items':[{'category':'meals','description':'Client dinner for 4 (business development)','amount':220.,'receipt':False}]}]

## 2. Grounded tools

The agent has four focused tools: policy lookup, receipt checking, limit checking, and approval-threshold checking. The tools return structured data so the decision can be inspected later.

In [3]:
ELIGIBLE_CATEGORIES = {
    "airfare",
    "lodging",
    "meals",
    "ground_transport",
    "conference_fees",
}

INELIGIBLE_CATEGORIES = {
    "alcohol",
    "minibar",
    "spa",
    "gym",
    "personal_entertainment",
    "in_room_movie",
    "personal_shopping",
    "gifts",
    "traffic_fine",
    "penalty",
    "late_fee",
    "personal_expense",
}


def trip_days(claim: dict) -> int:
    trip_start = date.fromisoformat(claim["trip_start"])
    trip_end = date.fromisoformat(claim["trip_end"])
    return (trip_end - trip_start).days + 1


def quantity_from_description(item: dict, unit: str, default: int) -> int:
    match = re.search(r"(\d+)\s*" + unit, item["description"].lower())
    return int(match.group(1)) if match else default


@tool
def policy_lookup(policy_ids: list[str]) -> dict:
    """Retrieve policy text by stable ID before explaining a decision."""
    return {
        policy_id: POLICIES.get(policy_id, "Policy not found.")
        for policy_id in policy_ids
    }


@tool
def receipt_checker(item: dict) -> dict:
    """Check whether a receipt is required and available for an item."""
    receipt_required = item["category"] in {"airfare", "lodging"} or item["amount"] > 25
    receipt_present = item["receipt"]

    return {
        "status": "OK" if not receipt_required or receipt_present else "MISSING_RECEIPT",
        "receipt_required": receipt_required,
        "receipt_present": receipt_present,
    }


@tool
def limit_checker(item: dict, trip_length_days: int) -> dict:
    """Apply meal, lodging, and ground-transport limits to one item."""
    category_limits = {
        "meals": (75, "days"),
        "lodging": (200, "nights"),
        "ground_transport": (50, "days"),
    }

    if item["category"] not in category_limits:
        return {"approved_amount": item["amount"], "deducted_amount": 0.0, "status": "NO_LIMIT"}

    daily_limit, unit = category_limits[item["category"]]
    allowed_amount = daily_limit * quantity_from_description(item, unit, trip_length_days)
    deducted_amount = max(0.0, item["amount"] - allowed_amount)

    return {
        "approved_amount": min(item["amount"], allowed_amount),
        "deducted_amount": deducted_amount,
        "status": "WITHIN_LIMIT" if deducted_amount == 0 else "OVER_LIMIT",
    }


@tool
def approval_threshold_checker(reimbursable_amount: float) -> dict:
    """Return the approval tier for the total amount after category caps."""
    if reimbursable_amount <= 500:
        return {"policy_ref": "POL-APR-01", "manual_review": False}
    if reimbursable_amount <= 2000:
        return {"policy_ref": "POL-APR-02", "manual_review": False}
    return {"policy_ref": "POL-APR-03", "manual_review": True}


TOOLS = [policy_lookup, receipt_checker, limit_checker, approval_threshold_checker]


## 3. Policy evaluation workflow

For each claim, the workflow checks category eligibility, receipts, category limits, airfare class, submission timeliness, and approval authority. Missing required receipts, policy exceptions, late claims, and high-value totals take priority and are routed to Manual Review.

The final decision and monetary calculations are deterministic. This avoids relying on an LLM for arithmetic or strict policy enforcement.

In [4]:
class ClaimResult(BaseModel):
    claim_id: str
    decision: Literal["APPROVE", "PARTIAL_APPROVE", "REJECT", "MANUAL_REVIEW"]
    approved_amount: float
    deducted_amount: float
    missing_docs: list[str]
    policy_refs: list[str]
    confidence: float = Field(ge=0, le=1)
    explanation: str
    tools_used: list[str]


def evaluate_claim(claim: dict) -> tuple[ClaimResult, dict]:
    total_approved = 0.0
    total_deducted = 0.0
    policy_refs = set()
    missing_docs = []
    manual_review_reasons = []
    ineligible_item_count = 0
    audit_checks = []

    for item_number, item in enumerate(claim["items"], start=1):
        receipt_result = receipt_checker.invoke({"item": item})
        limit_result = limit_checker.invoke({"item": item, "trip_length_days": trip_days(claim)})
        category = item["category"]

        audit_checks.append({"item": item, "receipt": receipt_result, "limit": limit_result})

        if category not in ELIGIBLE_CATEGORIES or category in INELIGIBLE_CATEGORIES:
            ineligible_item_count += 1
            total_deducted += item["amount"]
            policy_refs.add("POL-CAT-02")
        else:
            total_approved += limit_result["approved_amount"]
            total_deducted += limit_result["deducted_amount"]
            policy_refs.add("POL-CAT-01")

            if limit_result["deducted_amount"] > 0:
                limit_policy = {"meals": "POL-PD-01", "lodging": "POL-PD-02", "ground_transport": "POL-PD-03"}
                policy_refs.add(limit_policy[category])

        if receipt_result["status"] == "MISSING_RECEIPT":
            missing_docs.append(f"Item {item_number}: itemized receipt for {category}")
            manual_review_reasons.append("missing required receipt")
            policy_refs.update({"POL-RCT-01", "POL-RCT-02"})

        if category == "airfare" and any(word in item["description"].lower() for word in ("business", "first-class", "first class")):
            manual_review_reasons.append("non-economy airfare exception")
            policy_refs.add("POL-AIR-01")

    submitted_date = date.fromisoformat(claim["submitted"])
    trip_end_date = date.fromisoformat(claim["trip_end"])
    if (submitted_date - trip_end_date).days > 30:
        manual_review_reasons.append("late submission")
        policy_refs.add("POL-TIME-01")

    approval_tier = approval_threshold_checker.invoke({"reimbursable_amount": total_approved})
    policy_refs.add(approval_tier["policy_ref"])

    if approval_tier["manual_review"]:
        manual_review_reasons.append("director approval threshold")

    if manual_review_reasons:
        decision = "MANUAL_REVIEW"
        confidence = 0.98
        reasons = manual_review_reasons
    elif ineligible_item_count == len(claim["items"]):
        decision = "REJECT"
        confidence = 0.99
        reasons = ["all items are ineligible"]
    elif total_deducted > 0:
        decision = "PARTIAL_APPROVE"
        confidence = 0.99
        reasons = ["per-diem excess deducted"]
    else:
        decision = "APPROVE"
        confidence = 0.99
        reasons = ["all items comply with policy"]

    explanation = (
        f"{decision.replace('_', ' ').title()}: {'; '.join(reasons)}. "
        f"Approved ${total_approved:.2f}; deducted ${total_deducted:.2f}."
    )

    result = ClaimResult(
        claim_id=claim["claim_id"],
        decision=decision,
        approved_amount=round(total_approved, 2),
        deducted_amount=round(total_deducted, 2),
        missing_docs=missing_docs,
        policy_refs=sorted(policy_refs),
        confidence=confidence,
        explanation=explanation,
        tools_used=["policy_lookup", "receipt_checker", "limit_checker", "approval_threshold_checker"],
    )

    audit_record = {"claim": claim, "checks": audit_checks, "threshold": approval_tier}
    return result, audit_record


## 4. GenAI tool-calling 

In [5]:
def generate_grounded_explanation(result: ClaimResult, audit_record: dict) -> str | None:
    """Ask the LLM for an explanation after it has used the policy tool."""
    if not os.getenv("GROQ_API_KEY"):
        return None

    try:
        from langchain_groq import ChatGroq

        llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0).bind_tools(TOOLS)
        messages = [
            SystemMessage(content="You are a travel audit assistant. Call policy_lookup before explaining. Never change the supplied decision."),
            HumanMessage(content=json.dumps({"decision": result.model_dump(), "audit": audit_record}, default=str)),
        ]

        for _ in range(4):
            response = llm.invoke(messages)
            messages.append(response)

            if not response.tool_calls:
                return response.content.strip() or None

            for tool_call in response.tool_calls:
                selected_tool = next(tool for tool in TOOLS if tool.name == tool_call["name"])
                tool_result = selected_tool.invoke(tool_call["args"])
                messages.append(ToolMessage(content=json.dumps(tool_result), tool_call_id=tool_call["id"]))

    except Exception as error:
        print("LLM explanation unavailable:", type(error).__name__)

    return None


sample_result, sample_audit = evaluate_claim(CLAIMS[2])
print(generate_grounded_explanation(sample_result, sample_audit) or sample_result.explanation)


Partial Approve: per-diem excess deducted. Approved $840.00; deducted $100.00.


## 5. Evaluation and validation

The next cell evaluates all five supplied claims and performs assertions against expected policy outcomes. These checks make incorrect changes visible immediately. The table also provides sample output for all five claims.

In [6]:
evaluations = [evaluate_claim(claim) for claim in CLAIMS]
results = [result.model_dump() for result, _ in evaluations]

required_fields = {
    "claim_id",
    "decision",
    "approved_amount",
    "deducted_amount",
    "missing_docs",
    "policy_refs",
    "confidence",
    "explanation",
    "tools_used",
}
expected_decisions = [
    "APPROVE",
    "REJECT",
    "PARTIAL_APPROVE",
    "MANUAL_REVIEW",
    "MANUAL_REVIEW",
]

assert len(results) == 5
assert [result["decision"] for result in results] == expected_decisions
assert results[2]["approved_amount"] == 840.0
assert results[2]["deducted_amount"] == 100.0
assert all(set(result) == required_fields for result in results)

pd.DataFrame(results)[
    ["claim_id", "decision", "approved_amount", "deducted_amount", "missing_docs"]
]


,claim_id,decision,approved_amount,deducted_amount,missing_docs
0,CLM-001,APPROVE,1110.0,0.0,[]
1,CLM-002,REJECT,0.0,380.0,[]
2,CLM-003,PARTIAL_APPROVE,840.0,100.0,[]
3,CLM-004,MANUAL_REVIEW,3000.0,0.0,[Item 2: itemized receipt for lodging]
4,CLM-005,MANUAL_REVIEW,75.0,145.0,[Item 1: itemized receipt for meals]


## Dashboard

This dashboard is generated from the actual structured results, not hard-coded values. It shows approved and deducted amounts, the decision distribution, and claims requiring human attention.

In [7]:
dashboard = pd.DataFrame(results)

total_approved = dashboard['approved_amount'].sum()
total_deducted = dashboard['deducted_amount'].sum()
manual_reviews = (dashboard['decision'] == 'MANUAL_REVIEW').sum()

cards = f'''<div style='display:flex; gap:12px; margin:8px 0 20px 0;'>
<div style='padding:14px 18px; border-radius:10px; background:#eef4ff; min-width:150px'><b>Claims processed</b><br><span style='font-size:24px'>{len(dashboard)}</span></div>
<div style='padding:14px 18px; border-radius:10px; background:#eaf7ef; min-width:150px'><b>Approved amount</b><br><span style='font-size:24px'>$ {total_approved:,.0f}</span></div>
<div style='padding:14px 18px; border-radius:10px; background:#fff1f0; min-width:150px'><b>Deducted amount</b><br><span style='font-size:24px'>$ {total_deducted:,.0f}</span></div>
<div style='padding:14px 18px; border-radius:10px; background:#fff8e6; min-width:150px'><b>Manual reviews</b><br><span style='font-size:24px'>{manual_reviews}</span></div>
</div>'''
display(HTML(cards))

amount_chart = go.Figure()
amount_chart.add_bar(name='Approved', x=dashboard['claim_id'], y=dashboard['approved_amount'], marker_color='#238636')
amount_chart.add_bar(name='Deducted', x=dashboard['claim_id'], y=dashboard['deducted_amount'], marker_color='#d73a49')
amount_chart.update_layout(title='Reimbursement outcome by claim', barmode='stack', yaxis_title='USD', template='plotly_white', legend_title_text='')
amount_chart.show()

decision_counts = dashboard['decision'].value_counts().rename_axis('decision').reset_index(name='claims')
decision_chart = px.bar(decision_counts, x='decision', y='claims', text='claims', color='decision', title='Decision breakdown', color_discrete_map={'APPROVE':'#238636', 'PARTIAL_APPROVE':'#bf8700', 'REJECT':'#d73a49', 'MANUAL_REVIEW':'#8250df'})
decision_chart.update_layout(template='plotly_white', showlegend=False, xaxis_title='', yaxis_title='Number of claims')
decision_chart.show()

display(dashboard[['claim_id', 'decision', 'approved_amount', 'deducted_amount', 'missing_docs']].style.format({'approved_amount':'$ {:,.2f}', 'deducted_amount':'$ {:,.2f}'}))

,claim_id,decision,approved_amount,deducted_amount,missing_docs
0,CLM-001,APPROVE,"$ 1,110.00",$ 0.00,[]
1,CLM-002,REJECT,$ 0.00,$ 380.00,[]
2,CLM-003,PARTIAL_APPROVE,$ 840.00,$ 100.00,[]
3,CLM-004,MANUAL_REVIEW,"$ 3,000.00",$ 0.00,['Item 2: itemized receipt for lodging']
4,CLM-005,MANUAL_REVIEW,$ 75.00,$ 145.00,['Item 1: itemized receipt for meals']


## Design Notes & Reasoning

**Approach.** I used a hybrid design. Rule-sensitive operations—eligibility, receipt requirements, caps, and approval thresholds—are implemented in Python tools. The LLM uses those same tools for a grounded narrative explanation, but not for financial calculations.

**Manual-review choices.** Missing receipts, business/first-class airfare, late submissions, totals above $2,000, and conflicting information are routed to Manual Review. This follows the policy instruction to prefer human review in uncertain or exceptional cases.

**Assumptions and limitations.** The trip end date is used as the expense date for the 30-day check. When a description omits nights or days, the workflow uses the trip duration as a prototype fallback. A production system should capture these quantities explicitly.

**Trade-offs and next steps.** I chose a single-notebook implementation so the demo is easy to run and review. The LLM is optional, preventing external credentials from blocking the result. Future improvements would include receipt OCR, duplicate detection, persisted audit logs, a claim-entry form, and unit tests.

## 8. Final structured results

The final cell prints the required JSON array. Each object contains exactly these fields: claim_id, decision, approved_amount, deducted_amount, missing_docs, policy_refs, confidence, explanation, and tools_used.

In [8]:
print(json.dumps(results,indent=2))

[
  {
    "claim_id": "CLM-001",
    "decision": "APPROVE",
    "approved_amount": 1110.0,
    "deducted_amount": 0.0,
    "missing_docs": [],
    "policy_refs": [
      "POL-APR-02",
      "POL-CAT-01"
    ],
    "confidence": 0.99,
    "explanation": "Approve: all items comply with policy. Approved $1110.00; deducted $0.00.",
    "tools_used": [
      "policy_lookup",
      "receipt_checker",
      "limit_checker",
      "approval_threshold_checker"
    ]
  },
  {
    "claim_id": "CLM-002",
    "decision": "REJECT",
    "approved_amount": 0.0,
    "deducted_amount": 380.0,
    "missing_docs": [],
    "policy_refs": [
      "POL-APR-01",
      "POL-CAT-02"
    ],
    "confidence": 0.99,
    "explanation": "Reject: all items are ineligible. Approved $0.00; deducted $380.00.",
    "tools_used": [
      "policy_lookup",
      "receipt_checker",
      "limit_checker",
      "approval_threshold_checker"
    ]
  },
  {
    "claim_id": "CLM-003",
    "decision": "PARTIAL_APPROVE",
    "appro